In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.lines as mlines

In [ ]:
def render_timeline_panel(
    ax, data, periods, x_max, x_ticks, title, diff_col, colors,
    ticks=None, label_col='EEG_measure', label_formatter=None,
    marker_formatter=None, y_limit_padding=2.0, show_period_labels=True,
    bar_width=5.5, label_fontsize=10.5, label_fontweight='normal',
):
    """Render one timeline panel using one fixed row per result."""
    if data.empty:
        return

    ticks = ticks if ticks is not None else pd.DataFrame()
    label_formatter = label_formatter or (lambda row: str(row[label_col]))
    marker_formatter = marker_formatter or (lambda row: None)

    ax.set_xlim(0, x_max)
    ax.set_xticks(x_ticks)
    ax.set_ylim(-1, data['y_pos'].max() + max(y_limit_padding, 4.2))

    header_levels = [1.8, 3.0]
    for period_index, (start, end, period_color, period_title) in enumerate(periods):
        plot_end = min(end, x_max)
        if start >= x_max:
            continue
        ax.axvspan(start, plot_end, color=period_color, alpha=0.65, zorder=0)
        if show_period_labels:
            ax.text(
                (start + plot_end) / 2,
                data['y_pos'].max() + header_levels[period_index % 2],
                period_title,
                fontsize=9.5,
                fontweight='bold',
                color='#444444',
                ha='center',
                va='center',
                multialignment='center',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                          edgecolor='#cccccc', alpha=0.9),
            )

    if not ticks.empty:
        ax.set_yticks(ticks['y_tick'])
        ax.set_yticklabels(
            ticks['short_id'].astype(str).str.replace('_', ' ', regex=False),
            fontsize=10, fontweight='bold',
        )
        for _, boundary in ticks.iterrows():
            ax.axhline(boundary['y_top_divider'], color='gray', linestyle=':',
                       alpha=0.6, zorder=1)

    for _, row in data.iterrows():
        t_start = max(float(row['time_window_start']), 0)
        t_end = min(float(row['time_window_end']), x_max)
        if t_start >= t_end:
            continue

        diff_value = int(row[diff_col]) if pd.notna(row[diff_col]) else 0
        color = colors.get(diff_value, '#000000')
        y_pos = float(row['y_pos'])
        x_mid = (t_start + t_end) / 2
        ax.plot([t_start, t_end], [y_pos, y_pos], color=color,
                linewidth=bar_width, solid_capstyle='round', zorder=3)

        marker = marker_formatter(row)
        if marker is not None:
            ax.plot(x_mid, y_pos, marker=marker, markersize=7,
                    color='#2c3e50', linestyle='None', zorder=5)

        ax.text(
            x_mid, y_pos + 0.18, label_formatter(row),
            fontsize=label_fontsize, fontweight=label_fontweight,
            color='#222222', ha='center', va='bottom', zorder=4,
        )

    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Time post-stimulus (ms)', fontsize=10.5,
                  fontweight='bold', labelpad=6)
    ax.grid(True, axis='x', linestyle='--', alpha=0.5, color='#cccccc')
    ax.grid(False, axis='y')
    sns.despine(ax=ax, left=False, bottom=False, right=False)


def get_x_axis_limit(data, step):
    """Return the smallest step multiple containing all time windows."""
    if data.empty:
        return step
    maximum_time = pd.to_numeric(data['time_window_end'], errors='coerce').max()
    return max(step, int(np.ceil(maximum_time / step) * step))


def render_inhibition_timeline(ax, data, ticks, periods, x_max=800, title='Inhibition'):
    render_timeline_panel(ax=ax, data=data, ticks=ticks, periods=periods,
                          x_max=x_max, x_ticks=np.arange(0, x_max + 1, 100),
                          title=title, diff_col=diff_col, colors=colors,
                          label_col='EEG_measure', y_limit_padding=2.5)


def render_working_memory_timeline(ax, data, ticks, periods, x_max, title):
    """Working-memory wrapper with frequency labels and task markers."""
    def format_label(row):
        label = str(row['EEG_measure'])
        if row.get('is_frequency', False):
            min_freq = float(row['min_freq_hz'])
            max_freq = float(row['max_freq_hz'])
            min_text = int(min_freq) if min_freq.is_integer() else min_freq
            max_text = int(max_freq) if max_freq.is_integer() else max_freq
            label += f' ({min_text}-{max_text} Hz)'
        return label

    def marker_for_row(row):
        return 'D' if not row.get('is_nback', True) else None

    render_timeline_panel(ax=ax, data=data, ticks=ticks, periods=periods,
                          x_max=x_max,
                          x_ticks=np.arange(0, x_max + 1,
                                            200 if x_max <= 1000 else 500),
                          title=title, diff_col=diff_col, colors=colors,
                          label_formatter=format_label,
                          marker_formatter=marker_for_row)


def prepare_stacked_df(main_df):
    """Prepare fixed rows with extra vertical clearance for labels and dividers."""
    if main_df.empty:
        return pd.DataFrame(), pd.DataFrame(), 0

    prepared = main_df.copy()
    prepared['year_temp'] = prepared['study_id'].str.extract(r'(\d{4})').astype(float)
    prepared = prepared.sort_values(
        by=['year_temp', 'study_id', 'time_window_start'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    row_spacing = 1.5
    prepared['y_pos'] = np.arange(len(prepared), dtype=float) * row_spacing

    ticks_info = []
    for study, study_df in prepared.groupby('study_id', sort=False):
        min_y = study_df['y_pos'].min()
        max_y = study_df['y_pos'].max()
        ticks_info.append({
            'short_id': study_df['short_id'].iloc[0],
            'y_tick': (min_y + max_y) / 2,
            'y_top_divider': max_y + 0.7,
            'is_nback': study_df['is_nback'].all(),
        })

    total_height = len(prepared) * row_spacing
    return prepared.drop(columns='year_temp'), pd.DataFrame(ticks_info), total_height

In [ ]:
colors = {
    0: "#999999",  # No difference
    1: "#7570b3",  # Increased in ADHD / Compensatory
    2: "#d95f02"   # Decreased in ADHD / Deficit
}

# Shaded periods (Functional Epochs)
erp_periods = [
    (0, 150, '#f7fcf5', 'Sensory\n<150ms'),
    (150, 250, '#fff7bc', 'Early P2/N2\n(150-250ms)'),
    (250, 500, '#deebf7', 'Updating / P3\n(250-500ms)'),
    (500, 1000, '#f3e5f5', 'Late / Maintenance\n(>500ms)')
]

freq_periods = [
    (0, 200, '#f7fcf5', 'Sensory / Encoding\n(0-200 ms)'),
    (200, 500, '#fff7bc', 'Selection & Updating\n(200-500 ms)'),
    (500, 1000, '#deebf7', 'Maintenance & Storage\n(500-1000 ms)'),
    (1000, 2500, '#f3e5f5', 'Sustained Retention / Delay\n(>1000ms)')
]

In [ ]:
import matplotlib.gridspec as gridspec

# Generate the inhibition ERP timeline plot with a shared left x-axis
inhibition_df = pd.read_csv('data/inhibition_table.csv')
inhibition_diff_col = next(column for column in inhibition_df.columns if 'Difference' in column)
inhibition_include_col = next(column for column in inhibition_df.columns if 'include' in column.lower())
inhibition_df = inhibition_df[inhibition_df[inhibition_include_col] == 1].copy()
inhibition_df['time_window_start'] = pd.to_numeric(inhibition_df['time_window_start'], errors='coerce')
inhibition_df['time_window_end'] = pd.to_numeric(inhibition_df['time_window_end'], errors='coerce')
inhibition_df.dropna(subset=['time_window_start', 'time_window_end'], inplace=True)
inhibition_df['is_nback'] = inhibition_df['task'].astype(str).str.contains('back', case=False, na=False)

inhibition_periods = [
    (50, 150, '#f7fcf5', 'Early\n(50-150 ms)'),
    (150, 300, '#fff7bc', 'Conflict\n(150-300 ms)'),
    (300, 500, '#deebf7', 'Integration\n(300-500 ms)'),
    (500, 800, '#f3e5f5', 'Late Evaluation\n(500+ ms)'),
]

inhibition_tasks = [
    ('CPT_Go_NoGo', 'CPT_Go_NoGo'),
    ('stop_signal', 'Stop_Signal'),
    ('Go_NoGo', 'Go_NoGo'),
]
inhibition_panels = []
for task_name, task_title in inhibition_tasks:
    task_data = inhibition_df[
        inhibition_df['task'].astype(str).str.lower() == task_name.lower()
    ].copy()
    task_data, task_ticks, task_total_y = prepare_stacked_df(task_data)
    inhibition_panels.append((task_data, task_ticks, task_total_y, task_title))

inhibition_cpt, inhibition_sst, inhibition_gng = inhibition_panels
inhibition_left_panels = [inhibition_cpt, inhibition_sst]
inhibition_left_rows = sum(max(panel[2], 1) for panel in inhibition_left_panels)
inhibition_gng_rows = max(inhibition_gng[2], 1)

inhibition_fig = plt.figure(figsize=(18, max(9, inhibition_left_rows * 0.45)), dpi=150)
inhibition_grid = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.25)
inhibition_ax_left = inhibition_fig.add_subplot(inhibition_grid[0, 0])
inhibition_ax_gng = inhibition_fig.add_subplot(inhibition_grid[0, 1])

# Offset the two left-side tasks into one shared y-axis.
inhibition_left_data = []
inhibition_left_ticks = []
inhibition_left_offset = 0
for task_data, task_ticks, task_total_y, task_title in inhibition_left_panels:
    if not task_data.empty:
        task_data = task_data.copy()
        task_data['y_pos'] += inhibition_left_offset
        task_ticks = task_ticks.copy()
        task_ticks['y_tick'] += inhibition_left_offset
        task_ticks['y_top_divider'] += inhibition_left_offset
        inhibition_left_data.append(task_data)
        inhibition_left_ticks.append(task_ticks)
    inhibition_left_offset += max(task_total_y, 1)

inhibition_left_data = pd.concat(inhibition_left_data, ignore_index=True) if inhibition_left_data else pd.DataFrame()
inhibition_left_ticks = pd.concat(inhibition_left_ticks, ignore_index=True) if inhibition_left_ticks else pd.DataFrame()
inhibition_x_max = get_x_axis_limit(inhibition_left_data, step=100)
inhibition_gng_x_max = get_x_axis_limit(inhibition_gng[0], step=100)

render_timeline_panel(
    ax=inhibition_ax_left, data=inhibition_left_data,
    ticks=inhibition_left_ticks, periods=inhibition_periods,
    x_max=inhibition_x_max, x_ticks=np.arange(0, inhibition_x_max + 1, 100),
    title='', diff_col=inhibition_diff_col, colors=colors,
    y_limit_padding=2.5, show_period_labels=True, bar_width=5.0,
    label_fontsize=10.5, label_fontweight='medium',
)

inhibition_task_boundary = max(inhibition_cpt[2], 1) - 0.5
inhibition_ax_left.axhline(inhibition_task_boundary, color='#333333', linewidth=1.4, linestyle='-', zorder=2)
inhibition_ax_left.text(
    inhibition_x_max - 10, 0.15, 'CPT_Go_NoGo', ha='right', va='bottom',
    fontsize=13.5, fontweight='bold', color='#333333',
    bbox=dict(facecolor='white', edgecolor='#cccccc', alpha=0.85, pad=2), zorder=6,
)
inhibition_ax_left.text(
    inhibition_x_max - 10, inhibition_task_boundary + 0.15, 'Stop_Signal',
    ha='right', va='bottom', fontsize=13.5, fontweight='bold', color='#333333',
    bbox=dict(facecolor='white', edgecolor='#cccccc', alpha=0.85, pad=2), zorder=6,
)

render_timeline_panel(
    ax=inhibition_ax_gng, data=inhibition_gng[0], ticks=inhibition_gng[1],
    periods=inhibition_periods, x_max=inhibition_gng_x_max,
    x_ticks=np.arange(0, inhibition_gng_x_max + 1, 100), title='',
    diff_col=inhibition_diff_col, colors=colors, y_limit_padding=2.5,
    show_period_labels=True, bar_width=5.0,
    label_fontsize=10.5, label_fontweight='medium',
)
inhibition_ax_gng.text(
    inhibition_gng_x_max - 10, 0.15, 'Go_NoGo', ha='right', va='bottom',
    fontsize=13.5, fontweight='bold', color='#333333',
    bbox=dict(facecolor='white', edgecolor='#cccccc', alpha=0.85, pad=2), zorder=6,
)

inhibition_ax_gng.set_xlim(-50, inhibition_gng_x_max)

inhibition_legend = [
    mlines.Line2D([0], [0], color=colors[2], lw=5, label='Decreased in ADHD (Deficit)'),
    mlines.Line2D([0], [0], color=colors[1], lw=5, label='Increased in ADHD (Compensatory)'),
    mlines.Line2D([0], [0], color=colors[0], lw=5, label='No Difference'),
]
inhibition_fig.legend(
    handles=inhibition_legend, loc='upper center', bbox_to_anchor=(0.5, 0.075),
    ncol=3, frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=11,
)
inhibition_fig.tight_layout(rect=[0, 0.08, 1, 1])
inhibition_fig.savefig('plots/inhibtion_timeline.png', dpi=600, bbox_inches='tight')
plt.close(inhibition_fig)

In [ ]:
df = pd.read_csv('data/working_memory_table.csv')

# Detect relevant columns dynamically
diff_col = next(col for col in df.columns if 'Difference' in col)
include_col = next(col for col in df.columns if 'include' in col.lower())
df = df[df[include_col] == 1].copy()

# Ensure numeric time windows
df['time_window_start'] = pd.to_numeric(df['time_window_start'], errors='coerce')
df['time_window_end'] = pd.to_numeric(df['time_window_end'], errors='coerce')
df.dropna(subset=['time_window_start', 'time_window_end'], inplace=True)

# Split by measure type (Frequencies vs ERPs) and flag Non-N-Back tasks
df['is_frequency'] = df['min_freq_hz'].apply(
    lambda x: str(x).strip().upper() != 'NR' and pd.notna(x)
)
df['is_nback'] = df['task'].apply(
    lambda x: 'back' in str(x).lower()
)


df_erp_processed, ticks_erp, total_y_erp = prepare_stacked_df(df[~df['is_frequency']])
df_freq_processed, ticks_freq, total_y_freq = prepare_stacked_df(df[df['is_frequency']])
working_memory_erp_x_max = get_x_axis_limit(df_erp_processed, step=200)
working_memory_freq_x_max = get_x_axis_limit(df_freq_processed, step=500)


# Generate the working-memory ERP and frequency timeline plot
working_memory_fig, (working_memory_ax_erp, working_memory_ax_freq) = plt.subplots(
    1,
    2,
    figsize=(20, 11),
    dpi=150,
)

render_working_memory_timeline(
    ax=working_memory_ax_erp,
    data=df_erp_processed,
    ticks=ticks_erp,
    periods=erp_periods,
    x_max=working_memory_erp_x_max,
    title='ERP Amplitudes & Components',
)
render_working_memory_timeline(
    ax=working_memory_ax_freq,
    data=df_freq_processed,
    ticks=ticks_freq,
    periods=freq_periods,
    x_max=working_memory_freq_x_max,
    title='Time-Frequency & Spectral Oscillations',
)
working_memory_ax_freq.set_xlim(-250, working_memory_freq_x_max)

working_memory_legend = [
    mlines.Line2D(
        [0], [0], color=colors[2], lw=5,
        label='Decreased in ADHD (Deficit)',
    ),
    mlines.Line2D(
        [0], [0], color=colors[1], lw=5,
        label='Increased in ADHD (Compensatory)',
    ),
    mlines.Line2D(
        [0], [0], color=colors[0], lw=5,
        label='No Difference',
    ),
    mlines.Line2D(
        [0], [0], marker='D', color='w', markerfacecolor='#2c3e50',
        markersize=8, label='Non-N-Back Task (D)',
    ),
]
working_memory_fig.legend(
    handles=working_memory_legend,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.005),
    ncol=4,
    frameon=True,
    facecolor='white',
    edgecolor='#cccccc',
    fontsize=11,
)
working_memory_fig.tight_layout(rect=[0, 0.035, 1, 0.96])
working_memory_fig.savefig(
    'plots/wm_timeline.png',
    dpi=300,
    bbox_inches='tight',
)
plt.close(working_memory_fig)